# GBM予測 & 3銘柄ポートフォリオ

対話的UI:
1. **1銘柄GBM予測** — μ・σ調整可、±1〜3σシグマバンド、VaR 95%/99%
2. **銘柄検索ヘルパー** — 日本語・英語名・ティッカーで検索 → ボタンで銘柄リストに追加
3. **3銘柄ポートフォリオ** — 相関ベースの真のリスク、分散効果、ポートフォリオVaR

### 推定モードのデフォルト
`2年平均`（過去2年の60日ローリングμ・σの平均）。スライダー操作で自動的に手動モードに切替。
水位グラフのローリング窓は推定モードに連動。

※データは毎回Yahoo Financeから取得。外部ファイル不要。

## 1. セットアップ

In [ ]:
import logging, warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

from datetime import datetime, timedelta
import re
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', context='notebook', font='Hiragino Sans', rc={'axes.unicode_minus': False})

TRADING_DAYS = 252
RF = 0.0
DEFAULT_ROLL_WINDOW = 60
HISTORY_YEARS = 2

to_date = datetime.today()
from_date = to_date - timedelta(days=365 * (HISTORY_YEARS + 1))
FROM_STR = from_date.strftime('%Y-%m-%d')
TO_STR = to_date.strftime('%Y-%m-%d')
HISTORY_CUTOFF = pd.Timestamp(to_date) - pd.Timedelta(days=365 * HISTORY_YEARS)
print(f'取得期間: {from_date.date()} 〜 {to_date.date()}')

## 2. 指定30銘柄リスト（ノートブック内蔵）

In [ ]:
DESIGNATED_STOCKS = [
    (7974, "任天堂", "エンタメ", "ゲーム"),
    (9602, "東宝", "エンタメ", "コンテンツ"),
    (9766, "コナミ", "エンタメ", "ゲーム"),
    (9684, "スクウェア・エニックス・ホールディングス", "エンタメ", "ゲーム"),
    (4816, "東映アニメーション", "エンタメ", "コンテンツ"),
    (6758, "ソニーグループ", "エンタメ", "総合"),
    (9605, "東映", "エンタメ", "コンテンツ"),
    (9697, "カプコン", "エンタメ", "ゲーム"),
    (5032, "ANYCOLOR", "エンタメ", "コンテンツ"),
    (9601, "松竹", "エンタメ", "コンテンツ"),
    (9020, "東日本旅客鉄道", "運輸", "鉄道"),
    (9022, "東海旅客鉄道", "運輸", "鉄道"),
    (9023, "東京地下鉄", "運輸", "鉄道"),
    (9064, "ヤマトホールディングス", "運輸", "陸運"),
    (9101, "日本郵船", "運輸", "海運"),
    (9104, "商船三井", "運輸", "海運"),
    (9107, "川崎汽船", "運輸", "海運"),
    (9143, "SGホールディングス", "運輸", "陸運"),
    (9201, "日本航空", "運輸", "空運"),
    (9202, "ANAホールディングス", "運輸", "空運"),
    (9843, "ニトリ", "小売", "家具・雑貨"),
    (3092, "ZOZO", "小売", "衣類"),
    (7532, "パン・パシフィック・インターナショナルホールディングス", "小売", "雑貨"),
    (9983, "ファーストリテイリング", "小売", "衣類"),
    (7581, "サイゼリヤ", "小売", "外食"),
    (3382, "セブン＆アイ・ホールディングス", "小売", "総合"),
    (7564, "ワークマン", "小売", "衣類"),
    (2702, "日本マクドナルドホールディングス", "小売", "外食"),
    (8267, "イオン", "小売", "総合"),
    (2695, "くら寿司", "小売", "外食"),
]

stocks = pd.DataFrame(DESIGNATED_STOCKS, columns=['code', 'name', 'sector', 'sub_sector'])
stocks['ticker'] = stocks['code'].astype(str).str.zfill(4) + '.T'
print(f'指定銘柄数: {len(stocks)}')
stocks.head()

## 3. Yahoo Finance から株価データを取得

In [ ]:
def load_all_stock_data():
    global ticker_to_name, name_to_ticker_designated, close, returns, ALL_STOCKS
    ticker_to_name = dict(zip(stocks['ticker'], stocks['name']))
    name_to_ticker_designated = dict(zip(stocks['name'], stocks['ticker']))
    raw = yf.download(
        stocks['ticker'].tolist(),
        start=FROM_STR, end=TO_STR,
        auto_adjust=True, progress=False, group_by='ticker',
    )
    if isinstance(raw.columns, pd.MultiIndex):
        close = pd.concat(
            {ticker_to_name[t]: raw[t]['Close'] for t in stocks['ticker'] if t in raw.columns.get_level_values(0)},
            axis=1,
        ).dropna(how='all').sort_index()
    else:
        close = pd.DataFrame({ticker_to_name[stocks['ticker'].iloc[0]]: raw['Close']}).dropna(how='all').sort_index()
    returns = close.pct_change().dropna(how='all')
    ALL_STOCKS = list(returns.columns)
    return ALL_STOCKS

load_all_stock_data()
print(f'{len(ALL_STOCKS)}銘柄: 価格 {close.shape}, リターン {returns.shape}')

## 4. ユーティリティ

In [ ]:
def gbm_bands(S0, mu, sigma, horizon_days):
    dt = 1 / TRADING_DAYS
    t = np.arange(horizon_days + 1) * dt
    log_mean = (mu - sigma**2 / 2) * t
    log_std = sigma * np.sqrt(t)
    median = S0 * np.exp(log_mean)
    expected = S0 * np.exp(mu * t)
    bands = {k: (S0 * np.exp(log_mean - k * log_std), S0 * np.exp(log_mean + k * log_std)) for k in [1, 2, 3]}
    return t, median, expected, bands

def var_levels(S0, mu, sigma, horizon_days, confidences=(0.95, 0.99)):
    T = horizon_days / TRADING_DAYS
    log_mean_T = (mu - sigma**2 / 2) * T
    log_std_T = sigma * np.sqrt(T)
    out = {}
    for c in confidences:
        price = S0 * np.exp(log_mean_T + log_std_T * norm.ppf(1 - c))
        out[c] = {'price': price, 'loss_pct': 1 - price / S0, 'loss_jpy': S0 - price}
    return out

def rolling_mu_sigma(ret_series, window):
    mu = ret_series.rolling(window).mean() * TRADING_DAYS
    sigma = ret_series.rolling(window).std() * np.sqrt(TRADING_DAYS)
    return mu, sigma

WINDOW_OPTIONS = ['2年平均', '直近10日', '直近20日', '直近60日', '直近120日', '直近252日']

def window_for_mode(mode):
    if mode == '2年平均':
        return DEFAULT_ROLL_WINDOW
    return int(mode.replace('直近', '').replace('日', ''))

def estimate_mu_sigma_from_ret(r, mode):
    if mode == '2年平均':
        mu_roll, sigma_roll = rolling_mu_sigma(r, DEFAULT_ROLL_WINDOW)
        mu_roll = mu_roll[mu_roll.index >= HISTORY_CUTOFF]
        sigma_roll = sigma_roll[sigma_roll.index >= HISTORY_CUTOFF]
        return mu_roll.mean(), sigma_roll.mean()
    window = int(mode.replace('直近', '').replace('日', ''))
    recent = r.iloc[-window:]
    return recent.mean() * TRADING_DAYS, recent.std() * np.sqrt(TRADING_DAYS)

extra_cache = {}
name_cache = {}

def fetch_company_name(ticker):
    if ticker in name_cache:
        return name_cache[ticker]
    try:
        info = yf.Ticker(ticker).info
        name = info.get('longName') or info.get('shortName') or ticker
    except Exception:
        name = ticker
    name_cache[ticker] = name
    return name

def normalize_ticker(value):
    value = value.strip()
    if value in ALL_STOCKS:
        return name_to_ticker_designated.get(value, value)
    ticker = value.upper()
    if ticker.isdigit():
        return ticker.zfill(4) + '.T'
    if not ticker.endswith('.T'):
        if ticker.replace('.', '').isdigit():
            return ticker.split('.')[0].zfill(4) + '.T'
        return ticker + '.T'
    return ticker

def fetch_ticker_data(ticker):
    if ticker in extra_cache:
        return extra_cache[ticker]
    try:
        df = yf.download(ticker, start=FROM_STR, end=TO_STR, auto_adjust=True, progress=False)
        if df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        c = df['Close'].dropna() if 'Close' in df.columns else df.iloc[:, 0].dropna()
        if isinstance(c, pd.DataFrame):
            c = c.iloc[:, 0]
        r = c.pct_change().dropna()
        company = fetch_company_name(ticker)
        display_name = f'{ticker} {company}' if company != ticker else ticker
        result = (display_name, c, r)
        extra_cache[ticker] = result
        return result
    except Exception:
        return None

def resolve_stock(value):
    value = value.strip()
    if value in ALL_STOCKS:
        return value, close[value].dropna(), returns[value].dropna()
    ticker = normalize_ticker(value)
    return fetch_ticker_data(ticker)

def add_stock_to_memory(ticker, company_name=None):
    global ticker_to_name, name_to_ticker_designated, close, returns, ALL_STOCKS
    if company_name is None:
        company_name = fetch_company_name(ticker)
    if company_name in ALL_STOCKS:
        return False, f'すでに登録済み: {company_name}'
    data = fetch_ticker_data(ticker)
    if data is None:
        return False, f'データ取得失敗: {ticker}'
    _, c_series, r_series = data
    close[company_name] = c_series
    returns[company_name] = r_series
    ticker_to_name[ticker] = company_name
    name_to_ticker_designated[company_name] = ticker
    ALL_STOCKS.append(company_name)
    return True, f'追加: {ticker} {company_name}'

BAND_PALETTE = ['#fde68a', '#fdba74', '#fca5a5']

---

## 5. 銘柄検索ヘルパー

- ティッカー（`7203`, `7203.T`） → 会社名
- 英語名（`Toyota`, `Sony`） → ティッカー
- 日本語名（`住友ベ`, `トヨタ`, `リクルート`） → ティッカー
- 結果横の **📥追加** ボタンで銘柄リストに追加 → ドロップダウンに即反映

In [ ]:
def _search_yfinance(q):
    out = []
    try:
        results = yf.Search(q, max_results=15).quotes or []
        for r in results:
            sym = r.get('symbol', '')
            name = r.get('longname') or r.get('shortname') or ''
            if sym and sym.endswith('.T'):
                out.append((sym, name))
    except Exception:
        pass
    return out

def _search_yahoojp(q):
    out = []
    try:
        r = requests.get(
            'https://finance.yahoo.co.jp/search/',
            params={'query': q},
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=5,
        )
        if r.status_code == 200:
            codes = sorted(set(re.findall(r'/quote/(\d{4}\.T)', r.text)))
            for tk in codes[:10]:
                out.append((tk, fetch_company_name(tk)))
    except Exception:
        pass
    return out

def lookup(query):
    q = query.strip()
    if not q:
        return None
    if q in ALL_STOCKS:
        tk = name_to_ticker_designated[q]
        return [(tk, q + '（登録済み）', 'designated')]
    if q.replace('.T', '').replace('.', '').isdigit():
        tk = normalize_ticker(q)
        return [(tk, fetch_company_name(tk), 'ticker→name')]
    merged = {}
    for tk, name in _search_yfinance(q):
        merged[tk] = (name, 'yfinance')
    for tk, name in _search_yahoojp(q):
        if tk not in merged:
            merged[tk] = (name, 'Yahoo!JP')
    if not merged:
        return []
    return [(tk, name, src) for tk, (name, src) in merged.items()]

_dropdown_widgets_to_sync = []

def _sync_dropdowns():
    for w in _dropdown_widgets_to_sync:
        prev = w.value
        w.options = ALL_STOCKS
        if prev in ALL_STOCKS:
            w.value = prev

lookup_input = widgets.Text(value='', placeholder='例: 7203 / 4203.T / Toyota / 住友ベ / トヨタ', description='検索')
lookup_btn = widgets.Button(description='🔍 検索', button_style='primary')
lookup_results_box = widgets.VBox([])
lookup_status = widgets.Output()

def _do_lookup(_=None):
    with lookup_status:
        lookup_status.clear_output()
        results = lookup(lookup_input.value)
        if results is None:
            print('（検索ワードを入力してください）')
            lookup_results_box.children = []
            return
        if not results:
            print(f"❌ '{lookup_input.value}' に該当する銘柄が見つかりませんでした")
            lookup_results_box.children = []
            return
        print(f"検索: '{lookup_input.value}'  → {len(results)}件")
    rows = []
    for sym, name, src in results:
        label = widgets.HTML(value=f'<code>{sym}</code>  {name}  <span style="color:#888;font-size:11px">[{src}]</span>')
        btn = widgets.Button(description='📥追加', button_style='success', layout=widgets.Layout(width='90px'))
        status = widgets.HTML(value='')
        def _make_handler(_sym, _name, _status):
            def _h(_):
                ok, msg = add_stock_to_memory(_sym, _name)
                color = '#10b981' if ok else '#dc2626'
                _status.value = f'<span style="color:{color}">{msg}</span>'
                if ok:
                    _sync_dropdowns()
            return _h
        btn.on_click(_make_handler(sym, name, status))
        rows.append(widgets.HBox([label, btn, status]))
    lookup_results_box.children = rows

lookup_btn.on_click(_do_lookup)
lookup_input.on_submit(_do_lookup)

display(widgets.HBox([lookup_input, lookup_btn]), lookup_status, lookup_results_box)

---

## 6. 1銘柄 GBM予測（対話UI）

スライダー操作で自動的に手動モードに切替。推定モードを再選択すると推定値に戻る。
水位グラフのローリング窓は推定モードに連動。

In [28]:
def plot_single_forecast(stock, mode, horizon_days, mu_override, sigma_override, use_override):
    series = close[stock].dropna()
    S0 = series.iloc[-1]
    last_date = series.index[-1]
    r = returns[stock].dropna()

    mu_est, sigma_est = estimate_mu_sigma_from_ret(r, mode)
    mu = mu_override if use_override else mu_est
    sigma = sigma_override if use_override else sigma_est

    _, median, expected, bands = gbm_bands(S0, mu, sigma, horizon_days)
    var = var_levels(S0, mu, sigma, horizon_days)
    forecast_dates = pd.bdate_range(last_date, periods=horizon_days + 1)
    hist = series.iloc[-90:]

    win = window_for_mode(mode)
    mu_roll, sigma_roll = rolling_mu_sigma(r, win)
    mu_roll = mu_roll[mu_roll.index >= HISTORY_CUTOFF]
    sigma_roll = sigma_roll[sigma_roll.index >= HISTORY_CUTOFF]
    mu_2y_avg = mu_roll.mean()
    sigma_2y_avg = sigma_roll.mean()

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(3, 1, height_ratios=[3, 1, 1], hspace=0.5)

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(hist.index, hist.values, color='#0f172a', linewidth=1.8, label='実績')
    for (k, (lo, hi)), c in zip(bands.items(), BAND_PALETTE):
        ax.fill_between(forecast_dates, lo, hi, color=c, alpha=0.6, label=f'±{k}σ')
    ax.plot(forecast_dates, median, color='#1e293b', linestyle='--', linewidth=1.6, label='中央値')
    ax.plot(forecast_dates, expected, color='#2563eb', linestyle=':', linewidth=1.4, label='期待値')
    ax.axhline(var[0.95]['price'], color='#dc2626', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax.axhline(var[0.99]['price'], color='#7f1d1d', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax.annotate(f"VaR 95% : ¥{var[0.95]['price']:,.0f} ({var[0.95]['loss_pct']:.1%})",
                xy=(forecast_dates[-1], var[0.95]['price']), xytext=(8, 0), textcoords='offset points',
                color='#dc2626', va='center', fontsize=9, fontweight='bold')
    ax.annotate(f"VaR 99% : ¥{var[0.99]['price']:,.0f} ({var[0.99]['loss_pct']:.1%})",
                xy=(forecast_dates[-1], var[0.99]['price']), xytext=(8, 0), textcoords='offset points',
                color='#7f1d1d', va='center', fontsize=9, fontweight='bold')
    ax.axvline(last_date, color='gray', linewidth=0.6)
    used = '手動' if use_override else mode
    ax.set_title(
        f'{stock} — GBM予測（{horizon_days}営業日 ≒ {horizon_days/21:.1f}ヶ月）\n'
        f'現在値 ¥{S0:,.0f} / μ={mu:.1%} / σ={sigma:.1%} ({used})',
        fontsize=12, fontweight='bold'
    )
    ax.set_ylabel('株価 (円)')
    ax.legend(loc='upper left', fontsize=9, frameon=True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'¥{x:,.0f}'))

    ax_mu = fig.add_subplot(gs[1, 0])
    ax_mu.plot(mu_roll.index, mu_roll.values, color='#0ea5e9', linewidth=1.6)
    ax_mu.fill_between(mu_roll.index, mu_roll.values, 0, color='#0ea5e9', alpha=0.15)
    ax_mu.axhline(0, color='gray', linewidth=0.5)
    ax_mu.axhline(mu_2y_avg, color='#0ea5e9', linewidth=1.0, linestyle=':', label=f'2年平均 {mu_2y_avg:.1%}')
    ax_mu.axhline(mu, color='#dc2626', linewidth=1.5, linestyle='--', label=f'今使ってる値 {mu:.1%}')
    ax_mu.set_title(f'μ（年率）過去2年水位 — {win}日ローリング', fontsize=10, fontweight='bold')
    ax_mu.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_mu.legend(loc='upper left', fontsize=8)

    ax_sigma = fig.add_subplot(gs[2, 0])
    ax_sigma.plot(sigma_roll.index, sigma_roll.values, color='#dc2626', linewidth=1.6)
    ax_sigma.fill_between(sigma_roll.index, sigma_roll.values, 0, color='#dc2626', alpha=0.15)
    ax_sigma.axhline(sigma_2y_avg, color='#dc2626', linewidth=1.0, linestyle=':', label=f'2年平均 {sigma_2y_avg:.1%}')
    ax_sigma.axhline(sigma, color='#0f172a', linewidth=1.5, linestyle='--', label=f'今使ってる値 {sigma:.1%}')
    ax_sigma.set_title(f'σ（年率）過去2年水位 — {win}日ローリング', fontsize=10, fontweight='bold')
    ax_sigma.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_sigma.legend(loc='upper left', fontsize=8)

    plt.show()

stock_dd = widgets.Dropdown(options=ALL_STOCKS, value='パン・パシフィック・インターナショナルホールディングス' if 'パン・パシフィック・インターナショナルホールディングス' in ALL_STOCKS else ALL_STOCKS[0], description='銘柄')
mode_dd = widgets.Dropdown(options=WINDOW_OPTIONS, value='2年平均', description='推定モード')
horizon_sl = widgets.IntSlider(value=63, min=21, max=126, step=7, description='予測日数', continuous_update=False)
use_override_cb = widgets.Checkbox(value=False, description='μ/σ手動上書き')
mu_sl = widgets.FloatSlider(value=0.10, min=-1.0, max=1.5, step=0.05, description='μ(年率)', readout_format='.0%', continuous_update=False)
sigma_sl = widgets.FloatSlider(value=0.30, min=0.05, max=1.5, step=0.05, description='σ(年率)', readout_format='.0%', continuous_update=False)

def _on_slider(change):
    use_override_cb.value = True
def _on_mode(change):
    use_override_cb.value = False
mu_sl.observe(_on_slider, names='value')
sigma_sl.observe(_on_slider, names='value')
mode_dd.observe(_on_mode, names='value')

_dropdown_widgets_to_sync.append(stock_dd)

ui = widgets.VBox([
    widgets.HBox([stock_dd, mode_dd, horizon_sl]),
    widgets.HBox([use_override_cb, mu_sl, sigma_sl]),
])
out = widgets.interactive_output(plot_single_forecast, {
    'stock': stock_dd, 'mode': mode_dd, 'horizon_days': horizon_sl,
    'mu_override': mu_sl, 'sigma_override': sigma_sl, 'use_override': use_override_cb,
})
display(ui, out)

Output()

---

## 7. 3銘柄ポートフォリオ（対話UI）

- 銘柄1・銘柄2: 登録銘柄から選択
- **銘柄3: 任意銘柄OK**（ティッカー直接入力可）
- 上の検索で「📥追加」したら銘柄1/2のドロップダウンにも自動で出てくる

In [ ]:
def plot_portfolio(s1, s2, s3, w1, w2, w3, mode, horizon_days):
    res = resolve_stock(s3)
    if res is None:
        print(f"❌ 銘柄3 '{s3}' を取得できませんでした")
        return
    s3_name, s3_close, s3_ret = res

    raw_w = np.array([w1, w2, w3], dtype=float)
    if raw_w.sum() == 0:
        raw_w = np.ones(3)
    w = raw_w / raw_w.sum()

    ret_df = pd.concat({s1: returns[s1], s2: returns[s2], s3_name: s3_ret}, axis=1).dropna()
    names = [s1, s2, s3_name]

    if mode == '2年平均':
        r_used = ret_df[ret_df.index >= HISTORY_CUTOFF]
    else:
        window = int(mode.replace('直近', '').replace('日', ''))
        r_used = ret_df.iloc[-window:]

    mu_i = r_used.mean().values * TRADING_DAYS
    cov = r_used.cov().values * TRADING_DAYS
    corr = r_used.corr().values
    sigma_i = np.sqrt(np.diag(cov))
    mu_p = float(w @ mu_i)
    sigma_p = float(np.sqrt(w @ cov @ w))
    sigma_naive = float(w @ sigma_i)
    diversification = 1 - sigma_p / sigma_naive if sigma_naive > 0 else 0
    sharpe_p = (mu_p - RF) / sigma_p if sigma_p > 0 else np.nan

    S0 = 100.0
    _, median, expected, bands = gbm_bands(S0, mu_p, sigma_p, horizon_days)
    var = var_levels(S0, mu_p, sigma_p, horizon_days)
    last_date = ret_df.index[-1]
    forecast_dates = pd.bdate_range(last_date, periods=horizon_days + 1)

    hist_window = 90
    port_ret = (ret_df.iloc[-hist_window:] * w).sum(axis=1)
    hist_value = (1 + port_ret).cumprod()
    hist_value = hist_value / hist_value.iloc[-1] * S0

    win = window_for_mode(mode)
    port_ret_all = (ret_df * w).sum(axis=1).dropna()
    mu_p_roll, sigma_p_roll = rolling_mu_sigma(port_ret_all, win)
    mu_p_roll = mu_p_roll[mu_p_roll.index >= HISTORY_CUTOFF]
    sigma_p_roll = sigma_p_roll[sigma_p_roll.index >= HISTORY_CUTOFF]
    mu_p_2y = mu_p_roll.mean()
    sigma_p_2y = sigma_p_roll.mean()

    fig = plt.figure(figsize=(16, 16))
    gs = fig.add_gridspec(4, 3, height_ratios=[3, 2, 1, 1], hspace=0.55, wspace=0.35)

    ax_main = fig.add_subplot(gs[0, :])
    ax_main.plot(hist_value.index, hist_value.values, color='#0f172a', linewidth=1.8, label='実績(正規化)')
    for (k, (lo, hi)), c in zip(bands.items(), BAND_PALETTE):
        ax_main.fill_between(forecast_dates, lo, hi, color=c, alpha=0.6, label=f'±{k}σ')
    ax_main.plot(forecast_dates, median, color='#1e293b', linestyle='--', linewidth=1.6, label='中央値')
    ax_main.plot(forecast_dates, expected, color='#2563eb', linestyle=':', linewidth=1.4, label='期待値')
    ax_main.axhline(var[0.95]['price'], color='#dc2626', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax_main.axhline(var[0.99]['price'], color='#7f1d1d', linewidth=1.3, linestyle='-.', alpha=0.85)
    ax_main.annotate(f"VaR 95% : {var[0.95]['price']:.1f} ({var[0.95]['loss_pct']:.1%})",
                     xy=(forecast_dates[-1], var[0.95]['price']), xytext=(8, 0), textcoords='offset points',
                     color='#dc2626', va='center', fontsize=9, fontweight='bold')
    ax_main.annotate(f"VaR 99% : {var[0.99]['price']:.1f} ({var[0.99]['loss_pct']:.1%})",
                     xy=(forecast_dates[-1], var[0.99]['price']), xytext=(8, 0), textcoords='offset points',
                     color='#7f1d1d', va='center', fontsize=9, fontweight='bold')
    ax_main.axvline(last_date, color='gray', linewidth=0.6)
    title_names = ' + '.join([s1[:6], s2[:6], s3_name[:20]])
    ax_main.set_title(
        f'ポートフォリオ予測 — {title_names}\n'
        f'{horizon_days}営業日（≒{horizon_days/21:.1f}ヶ月）  ({mode})  '
        f'μ_p={mu_p:.1%}  σ_p={sigma_p:.1%}  Sharpe={sharpe_p:.2f}  分散効果={diversification:.1%}',
        fontsize=11, fontweight='bold'
    )
    ax_main.legend(loc='upper left', fontsize=9, frameon=True)
    ax_main.set_ylabel('ポートフォリオ価値（初期=100）')

    palette3 = ['#ef4444', '#3b82f6', '#10b981']
    short_names = [n[:14] for n in names]

    ax_w = fig.add_subplot(gs[1, 0])
    bars = ax_w.barh(short_names, w * 100, color=palette3)
    for bar, val in zip(bars, w * 100):
        ax_w.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontsize=9)
    ax_w.set_xlim(0, max(w * 100) * 1.3 + 5)
    ax_w.set_title('ウェイト', fontweight='bold')
    ax_w.invert_yaxis()

    ax_metrics = fig.add_subplot(gs[1, 1])
    x = np.arange(3)
    ax_metrics.bar(x - 0.2, mu_i * 100, width=0.4, color='#0ea5e9', label='μ(年率)')
    ax_metrics.bar(x + 0.2, sigma_i * 100, width=0.4, color='#dc2626', label='σ(年率)')
    ax_metrics.set_xticks(x)
    ax_metrics.set_xticklabels([n[:8] for n in names], fontsize=8, rotation=15)
    ax_metrics.set_ylabel('%')
    ax_metrics.legend(fontsize=8)
    ax_metrics.set_title('個別 μ / σ', fontweight='bold')
    ax_metrics.axhline(0, color='gray', linewidth=0.5)

    ax_corr = fig.add_subplot(gs[1, 2])
    sns.heatmap(
        pd.DataFrame(corr, index=[n[:8] for n in names], columns=[n[:8] for n in names]),
        annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
        ax=ax_corr, cbar_kws={'shrink': 0.7}, square=False, annot_kws={'fontsize': 10}
    )
    ax_corr.set_title('相関行列', fontweight='bold')

    ax_mu = fig.add_subplot(gs[2, :])
    ax_mu.plot(mu_p_roll.index, mu_p_roll.values, color='#0ea5e9', linewidth=1.6)
    ax_mu.fill_between(mu_p_roll.index, mu_p_roll.values, 0, color='#0ea5e9', alpha=0.15)
    ax_mu.axhline(0, color='gray', linewidth=0.5)
    ax_mu.axhline(mu_p_2y, color='#0ea5e9', linewidth=1.0, linestyle=':', label=f'2年平均 {mu_p_2y:.1%}')
    ax_mu.axhline(mu_p, color='#dc2626', linewidth=1.5, linestyle='--', label=f'今使ってる値 {mu_p:.1%}')
    ax_mu.set_title(f'μ_p（年率）過去2年水位 — 同ウェイトでの{win}日ローリング', fontsize=10, fontweight='bold')
    ax_mu.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_mu.legend(loc='upper left', fontsize=8)

    ax_sigma = fig.add_subplot(gs[3, :])
    ax_sigma.plot(sigma_p_roll.index, sigma_p_roll.values, color='#dc2626', linewidth=1.6)
    ax_sigma.fill_between(sigma_p_roll.index, sigma_p_roll.values, 0, color='#dc2626', alpha=0.15)
    ax_sigma.axhline(sigma_p_2y, color='#dc2626', linewidth=1.0, linestyle=':', label=f'2年平均 {sigma_p_2y:.1%}')
    ax_sigma.axhline(sigma_p, color='#0f172a', linewidth=1.5, linestyle='--', label=f'今使ってる値 {sigma_p:.1%}')
    ax_sigma.set_title(f'σ_p（年率）過去2年水位 — 同ウェイトでの{win}日ローリング', fontsize=10, fontweight='bold')
    ax_sigma.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax_sigma.legend(loc='upper left', fontsize=8)

    plt.show()

_default_s1 = 'パン・パシフィック・インターナショナルホールディングス' if 'パン・パシフィック・インターナショナルホールディングス' in ALL_STOCKS else ALL_STOCKS[0]
_default_s2 = '商船三井' if '商船三井' in ALL_STOCKS else ALL_STOCKS[1 if len(ALL_STOCKS) > 1 else 0]

ps1 = widgets.Dropdown(options=ALL_STOCKS, value=_default_s1, description='銘柄1')
ps2 = widgets.Dropdown(options=ALL_STOCKS, value=_default_s2, description='銘柄2')
ps3 = widgets.Text(value='4203.T', description='銘柄3', placeholder='ティッカー(例: 4203.T 住友ベークライト) or 指定銘柄名')
pw1 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w1', readout_format='.2f', continuous_update=False)
pw2 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w2', readout_format='.2f', continuous_update=False)
pw3 = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='w3', readout_format='.2f', continuous_update=False)
pmode = widgets.Dropdown(options=WINDOW_OPTIONS, value='2年平均', description='推定モード')
phorizon = widgets.IntSlider(value=63, min=21, max=126, step=7, description='予測日数', continuous_update=False)

_dropdown_widgets_to_sync.extend([ps1, ps2])

ui_p = widgets.VBox([
    widgets.HBox([ps1, pw1]),
    widgets.HBox([ps2, pw2]),
    widgets.HBox([ps3, pw3]),
    widgets.HBox([pmode, phorizon]),
])
out_p = widgets.interactive_output(plot_portfolio, {
    's1': ps1, 's2': ps2, 's3': ps3, 'w1': pw1, 'w2': pw2, 'w3': pw3,
    'mode': pmode, 'horizon_days': phorizon,
})
display(ui_p, out_p)

### 読み方
- **μ_p**: ポートフォリオ年率期待リターン（個別μの加重和）
- **σ_p**: 共分散行列ベースのポートフォリオ年率ボラ（相関を考慮した真のリスク）
- **分散効果**: $1 - \sigma_p / \sigma_{naive}$。値が大きいほど相関の低い銘柄を組合せている
- **VaR 95% / 99%**: 3ヶ月後の最悪損失水準
- **相関行列**: 0.5以上は同方向 → 分散効果が薄い。負相関はヘッジ効果
- **下段の水位グラフ**: いま使ってる値（赤破線）が過去の振れ幅のどこにいるかチェック

150万円制約に置き換えるなら、`VaR 95%の損失%` × 1.5M円 が「3ヶ月で覚悟する金額」。

---

## 8. リスク文言プロンプト生成

銘柄・購入額・VaR信頼水準などを選んで「✏️プロンプト生成」を押すと、ChatGPT/Claudeに投げられる **定量的リスク文言生成プロンプト** が出力される。
出力をコピーしてAIに貼れば、Gyosekiのノートに書く仮説・リスク記述がそのまま作れる。

In [ ]:
prompt_stock = widgets.Dropdown(options=ALL_STOCKS, value='パン・パシフィック・インターナショナルホールディングス' if 'パン・パシフィック・インターナショナルホールディングス' in ALL_STOCKS else ALL_STOCKS[0], description='銘柄')
prompt_conf = widgets.Dropdown(options=[('95%', 0.95), ('99%', 0.99)], value=0.95, description='VaR信頼水準')
prompt_amount = widgets.IntSlider(value=100, min=10, max=150, step=10, description='購入額(万円)', continuous_update=False)
prompt_horizon = widgets.IntSlider(value=63, min=21, max=126, step=7, description='保有期間(営業日)', continuous_update=False)
prompt_mode = widgets.Dropdown(options=WINDOW_OPTIONS, value='2年平均', description='推定モード')
gen_btn = widgets.Button(description='✏️ プロンプト生成', button_style='primary', layout=widgets.Layout(width='200px'))
prompt_output = widgets.Textarea(value='', layout=widgets.Layout(width='100%', height='400px'), placeholder='ここに生成されたプロンプトが入ります')
_dropdown_widgets_to_sync.append(prompt_stock)

def generate_risk_prompt(_=None):
    stock = prompt_stock.value
    conf = prompt_conf.value
    amount_yen = prompt_amount.value * 10000
    horizon = prompt_horizon.value
    mode = prompt_mode.value

    r = returns[stock].dropna()
    mu_est, sigma_est = estimate_mu_sigma_from_ret(r, mode)
    S0 = close[stock].dropna().iloc[-1]
    var_info = var_levels(S0, mu_est, sigma_est, horizon, (conf,))[conf]
    expected_loss_yen = var_info['loss_pct'] * amount_yen
    sharpe = (mu_est - RF) / sigma_est if sigma_est > 0 else 0
    ticker = name_to_ticker_designated.get(stock, normalize_ticker(stock))
    shares = int(amount_yen / S0) if S0 > 0 else 0

    prompt = (
        f'以下の定量データを踏まえて、ポートフォリオゲームの投資判断ノートに書ける「リスクに関する文言」を作成してください。\n'
        f'\n'
        f'## 投資対象\n'
        f'- 銘柄: {stock} ({ticker})\n'
        f'- 現在値: ¥{S0:,.0f}\n'
        f'- 購入予定額: ¥{amount_yen:,} ({prompt_amount.value}万円)\n'
        f'- 想定購入株数: 約{shares}株\n'
        f'- 保有期間: {horizon}営業日（≒{horizon/21:.1f}ヶ月）\n'
        f'\n'
        f'## ボラ・リターン指標（推定: {mode}）\n'
        f'- 年率期待リターン (μ): {mu_est:.1%}\n'
        f'- 年率ボラティリティ (σ): {sigma_est:.1%}\n'
        f'- シャープレシオ: {sharpe:.2f}\n'
        f'\n'
        f'## VaR (Value at Risk)\n'
        f'- 信頼水準: {conf:.0%}\n'
        f'- {horizon}営業日後の最悪想定価格: ¥{var_info["price"]:,.0f}\n'
        f'- 最悪損失率: {var_info["loss_pct"]:.1%}\n'
        f'- 購入予定額に対する最悪損失金額: ¥{expected_loss_yen:,.0f}\n'
        f'\n'
        f'## 要件\n'
        f'- 上記の数値を必ず引用しながら、定量的なリスク文言を **3〜5行** で書いてください\n'
        f'- 「{stock}株を{prompt_amount.value}万円分購入すると、{horizon/21:.0f}ヶ月後に {conf:.0%}の確率で最大 ¥{expected_loss_yen:,.0f} ({var_info["loss_pct"]:.1%}) の損失が発生し得る」というニュアンスを含めてください\n'
        f'- ボラとシャープレシオに触れて、リスクの妥当性に対するコメントも入れてください\n'
        f'- 投資仮説ノートに貼り付けられる、簡潔で読みやすい日本語で\n'
    )
    prompt_output.value = prompt

gen_btn.on_click(generate_risk_prompt)

display(widgets.VBox([
    widgets.HBox([prompt_stock, prompt_conf]),
    widgets.HBox([prompt_amount, prompt_horizon, prompt_mode]),
    gen_btn,
    prompt_output,
]))